In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.models import efficientnet_b0
from torchvision.models.efficientnet import SqueezeExcitation
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,  roc_auc_score,
    cohen_kappa_score, matthews_corrcoef, confusion_matrix, average_precision_score, fbeta_score,
    precision_recall_curve, balanced_accuracy_score)
import pandas as pd
from sklearn.utils import shuffle
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import torch_optimizer
import cv2
from skimage import measure
from torchvision.utils import make_grid
%matplotlib inline

In [ ]:
# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
!nvidia-smi

In [ ]:
file_name = "Benchmark_ConvNeXt_CV"


In [ ]:
import os

# Define directories of where the datasets reside
base_dir = '/home/pxk409/skin_lesions_13k'

# Define directories of where the training, validation and test sets reside
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Define directories of where the benign and malignant images reside
train_benign_dir = os.path.join(base_dir, 'train/benign')
train_malignant_dir = os.path.join(base_dir, 'train/malignant')
test_benign_dir = os.path.join(base_dir, 'test/benign')
test_malignant_dir = os.path.join(base_dir, 'test/malignant')

In [ ]:
# Let's check the number of images in each set
print('Total training benign images:', len(os.listdir(train_benign_dir)))
print('Total training malignant images:', len(os.listdir(train_malignant_dir)))
print('Total test benign images:', len(os.listdir(test_benign_dir)))
print('Total test malignant images:', len(os.listdir(test_malignant_dir)))

In [ ]:
# Load full dataset
full_dataset = datasets.ImageFolder(root=train_dir)

# Extract labels from the dataset
#labels = [label for _, label in full_dataset.samples]

# Stratified split (80% train, 20% validation)
#split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=123456)
#train_idx, val_idx = next(split.split(np.zeros(len(labels)), labels))

# Create training and validation subsets
#train_dataset = Subset(full_dataset, train_idx)
#val_dataset = Subset(full_dataset, val_idx)

#print(f"Training set: {len(train_idx)} samples")
#print(f"Validation set: {len(val_idx)} samples")

# Count the number of samples per class in each split
#train_classes = [full_dataset.targets[idx] for idx in train_dataset.indices]
#val_classes = [full_dataset.targets[idx] for idx in val_dataset.indices]

#print(f"Training set class counts: {np.bincount(train_classes)}")
#print(f"Validation set class counts: {np.bincount(val_classes)}")

In [ ]:
# Custom transform to integrate Albumentations with PyTorch
class AlbumentationsTransform:
    def __init__(self, augmentations):
        self.augmentations = A.Compose(augmentations)

    def __call__(self, image):
        image = np.array(image)  # Convert PIL Image to NumPy array
        augmented = self.augmentations(image=image)
        return augmented["image"]

# Simple augmentations for training
simple_transform = AlbumentationsTransform([
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=1.0),  # Random crop and resize
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.5),  # Vertical flip
    #A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),  # Color jitter
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2(),  # Convert to PyTorch tensor and ensure dtype=torch.float32
])


# Use AlbumentationsTransform directly in the loaders for consistency
train_transform = simple_transform

# Albumentations augmentations for validation and test
albumentations_val_test_transform = AlbumentationsTransform([
    A.Resize(height=224, width=224),  # Resize to EfficientNet input size
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2()  # Convert to PyTorch tensor
])

test_val_transform = albumentations_val_test_transform


In [ ]:
class TransformDataset(Dataset):
    def __init__(self, dataset, transform):
        """
        Args:
            dataset: Original dataset (e.g., Subset of ImageFolder).
            transform: Transform to be applied to the images in the dataset.
        """
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.dataset)

In [ ]:
# Wrap subsets with the respective transforms
#train_dataset = TransformDataset(train_dataset, train_transform)
#val_dataset = TransformDataset(val_dataset, test_val_transform)

In [ ]:
# Load and prepare the test dataset
#test_dataset = datasets.ImageFolder(root=test_dir, transform=test_val_transform)
test_dataset = datasets.ImageFolder(root=test_dir)
test_dataset = TransformDataset(test_dataset, test_val_transform)

In [ ]:
# Create data loaders
batch_size = 64
#num_cpus = multiprocessing.cpu_count()  # Use all CPU cores
#print(f"Number of CPU cores: {num_cpus}")
#num_workers = 16


#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
#train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
#val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
#test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

In [ ]:
def initialize_model():
    # Load pre-trained ConvNeXt-Tiny
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

    # Freeze the convolutional part (feature extractor)
    for param in model.features.parameters():
        param.requires_grad = False

    # Replace the classifier with a custom one
    # ConvNeXt-Tiny's flattened feature size is 768
    model.classifier = nn.Sequential(
        nn.Flatten(start_dim=1),  # Flatten the output [batch_size, 768, 1, 1] -> [batch_size, 768]
        nn.Linear(768, 512),   # Custom fully connected layer
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)      # Output layer for binary classification
    )

    # Ensure the classifier remains trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    # Move model to device
    model = model.to(device)
    return model

model = initialize_model()

In [ ]:
# Function to count trainable and non-trainable parameters
def count_trainable_non_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable

# Count parameters in the convolutional part
conv_trainable, conv_non_trainable = count_trainable_non_trainable_parameters(model.features)
print(f"Convolutional part (features):\n"
      f"  Trainable parameters: {conv_trainable:,}\n"
      f"  Non-trainable parameters: {conv_non_trainable:,}")

# Count parameters in the classifier part
classifier_trainable, classifier_non_trainable = count_trainable_non_trainable_parameters(model.classifier)
print(f"Classifier part:\n"
      f"  Trainable parameters: {classifier_trainable:,}\n"
      f"  Non-trainable parameters: {classifier_non_trainable:,}")

# Count parameters in the entire model
total_trainable, total_non_trainable = count_trainable_non_trainable_parameters(model)
print(f"Whole model:\n"
      f"  Trainable parameters: {total_trainable:,}\n"
      f"  Non-trainable parameters: {total_non_trainable:,}")

In [ ]:
# Early Stopping
class EarlyStopping:
    def __init__(self, patience, delta, path):
        """
        Args:
            patience (int): Number of epochs to wait after last improvement.
            delta (float): Minimum change in monitored metric to qualify as an improvement.
            path (str): Path to save the best model.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        score = -val_loss  # Early stopping is based on minimizing validation loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        """Saves model when validation loss decreases."""
        print(f"Validation loss decreased. Saving model to {self.path}")
        torch.save(model.state_dict(), self.path)


# Set the path for saving best_model.pth
#current_dir = os.getcwd()
#model_save_path = os.path.join(current_dir, 'best_model2.pth')

# Initialize EarlyStopping
#early_stopping = EarlyStopping(patience=5, delta=0.001, path=model_save_path)

In [ ]:
# Helper function to evaluate a dataset
def evaluate_model(model, data_loader, device):
    model.eval()  # Set model to evaluation mode
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1]  # Probabilities for the positive class
            _, preds = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# Helper function to calculate selected metrics
def calculate_metrics(labels, preds, probs):
    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds)
    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    f2 = fbeta_score(labels, preds, beta=2)
    auc = roc_auc_score(labels, probs)
    specificity = recall_score(labels, preds, pos_label=0)
    npv = precision_score(labels, preds, pos_label=0)

    return recall, precision, accuracy, f1, f2, auc, specificity, npv

In [ ]:
# Training loop with 5-fold cross-validation
cv_results = []
cv_metrics = []
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=123456)

# Extract labels from the dataset
labels = [label for _, label in full_dataset.samples]

# Cross-validation loop
for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n========== Fold {fold} ==========\n")

    # Create training and validation subsets for this fold
    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)

    # Apply transformations
    train_dataset = TransformDataset(train_dataset, train_transform)
    val_dataset = TransformDataset(val_dataset, test_val_transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Initialize model and optimizer for each fold
    model = initialize_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # EarlyStopping initialization
    early_stopping = EarlyStopping(patience=10, delta=0.0001, path=os.path.join(os.getcwd(), f"{file_name}_fold_{fold}.pth"))

    # Training loop
    num_epochs = 100
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels)

        train_loss /= len(train_loader)
        train_accuracy = train_correct.double() / len(train_dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels)

        val_loss /= len(val_loader)
        val_accuracy = val_correct.double() / len(val_dataset)
        val_losses.append(val_loss)

        print(f"Epoch {epoch + 1}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

        # Call EarlyStopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered. Ending training.")
            break

    # Store results for this fold
    cv_results.append({
        "fold": fold,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy.item(),
        "val_loss": val_loss,
        "val_accuracy": val_accuracy.item()
        })

    # Load the best model for evaluation
    model.load_state_dict(torch.load(f"{file_name}_fold_{fold}.pth", weights_only=True))

    # Evaluate on validation set
    labels, preds, probs = evaluate_model(model, val_loader, device)
    metrics = calculate_metrics(labels, preds, probs)
    recall, precision, accuracy, f1, f2, auc, specificity, npv = metrics

    # Record metrics for the fold
    cv_metrics.append({
        "Fold": fold,
        "Recall": recall,
        "Precision": precision,
        "Accuracy": accuracy,
        "F1 Score": f1,
        "F2 Score": f2,
        "AUC": auc,
        "Specificity": specificity,
        "NPV": npv,
    })

# Summarize cross-validation results
df_cv_results = pd.DataFrame(cv_results)
print("\n========== Cross-Validation Results ==========\n")
print(df_cv_results)

print("\nAverage Validation Accuracy: {:.4f}".format(df_cv_results["val_accuracy"].mean()))

In [ ]:
# Convert results to DataFrame
cv_metrics_df = pd.DataFrame(cv_metrics)

# Calculate average metrics
avg_metrics = cv_metrics_df.mean().to_dict()
avg_metrics["Fold"] = "Average"

# Add the average metrics to the DataFrame
cv_metrics_df = cv_metrics_df.append(avg_metrics, ignore_index=True)

# Save results to CSV file
csv_file_name = f"CV_Results/{file_name}.csv"
cv_metrics_df.to_csv(csv_file_name, index=False)

print(f"Cross-validation metrics saved to {csv_file_name}")